## Regenerating training_df

In [ ]:
import pandas as pd
import re
import numpy as np
def convert_clock_to_seconds(clock_str):
    match = re.match(r"PT(\d+)M(\d+)", clock_str)
    if match:
        minutes = int(match.group(1))
        seconds = int(match.group(2))
    
        return minutes * 60 + seconds
    
df = pd.read_csv("../data/pbp2024.csv")

gameids = df['gameid'].unique()
all_games_scoring = []

for game in gameids:
    single_game = df[df['gameid'] == game]

    # Keep ALL scoring rows first (including OT)
    full_scoring = single_game[single_game['h_pts'].notna()].copy()

    if full_scoring.empty:
        continue

    # Determine actual game winner from final score
    final_diff = ( full_scoring['h_pts'].iloc[-1] - full_scoring['a_pts'].iloc[-1] )

    home_win_bool = 1 if final_diff > 0 else 0

    # Now remove OT rows for feature generation
    scoring = full_scoring[full_scoring['period'] <= 4].copy()

    scoring['score_diff'] = scoring['h_pts'] - scoring['a_pts']
    scoring['quarter_seconds_remaining'] = ( scoring['clock'].apply(convert_clock_to_seconds) )

    scoring['game_seconds_remaining'] = ((4 - scoring['period']) * 720 + scoring['quarter_seconds_remaining'])

    scoring['home_win_bool'] = home_win_bool

    all_games_scoring.append(scoring[['gameid', 'score_diff', 'game_seconds_remaining', 'home_win_bool']])


training_df = pd.concat(
    all_games_scoring,
    ignore_index=True
)

## Random Forest Win Probability Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier

## Train-Test Split By Game

In [ ]:
gameids = training_df['gameid'].unique()
train_games, test_games = train_test_split(gameids, test_size=0.2, random_state=42)

train_df = training_df[training_df['gameid'].isin(train_games)]
test_df = training_df[training_df['gameid'].isin(test_games)]

print("train shape:" ,train_df.shape)
print("test shape:" ,test_df.shape)

print("Train games:", train_df['gameid'].nunique())
print("Test games:", test_df['gameid'].nunique())


Rows from the same game must not appear in both training and testing sets. Otherwise, the model could learn from nearly identical game states and produce overly optimistic performance estimates.

In [ ]:
x_train = train_df[['score_diff', 'game_seconds_remaining', 'interaction']]
y_train = train_df['home_win_bool']

x_test = test_df[['score_diff', 'game_seconds_remaining', 'interaction']]
y_test = test_df['home_win_bool']

model = RandomForestClassifier( n_estimators=100, random_state=42 )

# Train the model
model.fit(x_train, y_train)

# Make predictions
rf_predictions = model.predict(x_test)

# Evaluate accuracy
rf_accuracy = accuracy_score(
    y_test,
    rf_predictions
)

print("Random Forest Accuracy:", rf_accuracy)

## Win Probability Function

In [ ]:
def rf_win_probability(score_diff, time_remaining):
    interaction = score_diff * time_remaining

    return model.predict_proba(
        [[
            score_diff,
            time_remaining,
            interaction
        ]]
    )[0][1]

## Test a few common game situations

In [ ]:
print(rf_win_probability(0, 300))    # Tie game, 5 min left
print(rf_win_probability(5, 300))    # Home +5, 5 min left
print(rf_win_probability(5, 30))     # Home +5, 30 sec left
print(rf_win_probability(10, 30))    # Home +10, 30 sec left
print(rf_win_probability(-10, 30))   # Away +10, 30 sec left

## Display feature importance

In [ ]:
print(model.feature_importances_)

The Random Forest model produced highly confident predictions in late-game situations, often assigning probabilities of exactly 0 or 1. Despite its greater flexibility, it achieved lower test accuracy than Logistic Regression, suggesting that smoother decision boundaries may better capture NBA win probability dynamics.